In [1]:
!nvidia-smi

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1e9,
            2
        )
    )
else:
    print("No GPU detected. Check the Colab runtime settings.")

Mon Oct 12 17:17:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U transformers accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 113.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.6 MB/s eta 0:00:00


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-8B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

print("Loading quantized model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.eval()

print("\nModel loaded successfully.")
print("Model:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory allocated (GB):",
    round(torch.cuda.memory_allocated(0) / 1e9, 2),
)

Loading tokenizer...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


Model loaded successfully.
Model: Qwen/Qwen3-8B
GPU: Tesla T4
GPU memory allocated (GB): 6.09


In [4]:
messages = [
    {
        "role": "user",
        "content": "What is 20% of 100? Explain briefly."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(answer)

20% of 100 is calculated by multiplying 100 by 0.20.  
**20% of 100 = 20.**


In [5]:
import torch

sample = {
    "sample_id": "smoke_test_001",
    "question": "What is 20% of 100?",
    "steps": [
        "The original number is 100.",
        "20% means 20 divided by 100, which is 0.20.",
        "Multiplying 100 by 0.20 gives 20, so the answer is 20."
    ]
}

# Combine the question and reasoning trace.
prefix = f"Question: {sample['question']}\nReasoning:\n"
step_texts = [f"Step {i + 1}: {s}\n" for i, s in enumerate(sample["steps"])]
full_text = prefix + "".join(step_texts)

# Tokenize with offsets to map tokens to characters.
encoded = tokenizer(
    full_text,
    return_tensors="pt",
    return_offsets_mapping=True,
    add_special_tokens=True,
)

offsets = encoded.pop("offset_mapping")[0]
inputs = {k: v.to(model.device) for k, v in encoded.items()}

# Extract hidden states without generating additional text.
with torch.inference_mode():
    result = model(
        **inputs,
        output_hidden_states=True,
        use_cache=False,
    )

# Final transformer layer: [1, sequence_length, hidden_size].
token_hidden_states = result.hidden_states[-1][0].detach().cpu()

# Identify the character boundaries of each reasoning step.
step_boundaries = []
cursor = len(prefix)

for text in step_texts:
    start = cursor
    end = cursor + len(text)
    step_boundaries.append((start, end))
    cursor = end

# Map each step to overlapping tokens and mean-pool its representation.
token_offsets = offsets.tolist()
step_token_indices = []
step_hidden_states = []

for start, end in step_boundaries:
    indices = [
        i for i, (a, b) in enumerate(token_offsets)
        if b > a and a < end and b > start
    ]

    if not indices:
        raise ValueError(f"No tokens mapped to step boundary {(start, end)}")

    step_token_indices.append(indices)
    step_hidden_states.append(token_hidden_states[indices].mean(dim=0))

step_hidden_states = torch.stack(step_hidden_states)

print("Sample:", sample["sample_id"])
print("Number of tokens:", token_hidden_states.shape[0])
print("Token hidden-state shape:", tuple(token_hidden_states.shape))
print("Step hidden-state shape:", tuple(step_hidden_states.shape))
print("Step token counts:", [len(x) for x in step_token_indices])
print("Step boundaries:", step_boundaries)

Sample: smoke_test_001
Number of tokens: 86
Token hidden-state shape: (86, 4096)
Step hidden-state shape: (3, 4096)
Step token counts: [13, 27, 29]
Step boundaries: [(41, 77), (77, 129), (129, 192)]


In [6]:
from pathlib import Path

output_dir = Path("/content/retrace_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

artifact = {
    "sample_id": sample["sample_id"],
    "question": sample["question"],
    "steps": sample["steps"],
    "token_ids": encoded["input_ids"][0].cpu(),
    "token_offsets": token_offsets,
    "step_boundaries": step_boundaries,
    "step_token_indices": step_token_indices,
    "token_hidden_states": token_hidden_states,
    "step_hidden_states": step_hidden_states,
    "model_name": MODEL_NAME,
    "hidden_state_layer": -1,
    "pooling": "mean",
    "quantization": "4-bit NF4",
}

output_path = output_dir / "smoke_test.pt"
torch.save(artifact, output_path)

print("Saved:", output_path)
print("File size (MB):", round(output_path.stat().st_size / 1e6, 2))

Saved: /content/retrace_outputs/smoke_test.pt
File size (MB): 0.73


In [7]:
saved = torch.load(
    "/content/retrace_outputs/smoke_test.pt",
    map_location="cpu",
    weights_only=False,
)

assert len(saved["steps"]) == 3
assert saved["step_hidden_states"].shape[0] == 3
assert saved["token_hidden_states"].shape[0] == len(saved["token_ids"])

print("Artifact verification passed.")

Artifact verification passed.


In [8]:
from google.colab import files

files.download("/content/retrace_outputs/smoke_test.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>